<a href="https://colab.research.google.com/github/babi00/ai4biological-pattern/blob/guido-clean/invasive_plants_analysis/predictions_masked_regions_with_logits.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Extracting the predictions from the masked regions with the logits and the probabilities.

Built on top of final_model_evalutation.ipynb

In [ ]:
#@title Imports and downloads
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.transforms import functional as TF
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, ConcatDataset
from torchvision import transforms, models
# from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, multilabel_confusion_matrix, confusion_matrix, ConfusionMatrixDisplay, f1_score
# import seaborn as sns
# import requests
# from io import BytesIO
from google.colab import drive
import time
from tqdm import tqdm
!pip install open_clip_torch
import open_clip
# import random
import math
from collections import Counter
import datetime
import json
import os
import random


random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

drive.mount('/content/drive')

# Clone the repository and checkout the 'clean-barbara' branch
!git clone --branch clean-barbara https://github.com/babi00/ai4biological-pattern.git
%cd ai4biological-pattern

# Enable sparse checkout
!git sparse-checkout init --cone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.5 MB/s eta 0:00:00
Mounted at /content/drive
Cloning into 'ai4biological-pattern'...
remote: Enumerating objects: 735248, done.
remote: Counting objects: 100% (179/179), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 735248 (delta 94), reused 143 (delta 71), pack-reused 735069 (from 3)
Receiving objects: 100% (735248/735248), 8.79 GiB | 41.06 MiB/s, done.
Resolving deltas: 100% (2298/2298), done.
Updating files: 100% (214837/214837), done.
/content/ai4biological-pattern
Updating files: 100% (214835/214835), done.


In [ ]:
#@title Training log saving function
def save_training_log(model_name, output_dir, train_losses, val_losses, train_accuracies, val_accuracies, classification_report_dict):
    import datetime
    import json
    import os

    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_dir = f"/content/drive/MyDrive/Thesis/{output_dir}/logs"
    os.makedirs(log_dir, exist_ok=True)
    log_path = os.path.join(log_dir, "training_log.json")

    # Load existing logs (if any)
    if os.path.exists(log_path):
        with open(log_path, "r") as f:
            all_logs = json.load(f)
    else:
        all_logs = []

    # Prepare this run's data
    this_log = {
        "model_name": model_name,
        "timestamp": timestamp,
        "final_train_loss": train_losses[-1] if train_losses else None,
        "final_val_loss": val_losses[-1] if val_losses else None,
        "final_train_accuracy": train_accuracies[-1] if train_accuracies else None,
        "final_val_accuracy": val_accuracies[-1] if val_accuracies else None,
        "classification_report_last_epoch": classification_report_dict[-1] if isinstance(classification_report_dict, list) else classification_report_dict,
        "train_loss_history": train_losses,
        "val_loss_history": val_losses,
        "train_accuracy_history": train_accuracies,
        "val_accuracy_history": val_accuracies
    }

    all_logs.append(this_log)

    # Save updated list
    with open(log_path, "w") as f:
        json.dump(all_logs, f, indent=4)

    print(f"📝 Appended log to {log_path}")


In [ ]:
#@title Original Dataset
class InvasiveSpeciesDataset(Dataset):
    def __init__(self, root_dir, transform=None, get_label_fn=None):
        self.root_dir = root_dir
        self.transform = transform
        self.entries = []
        self.label_map = {
            "Invasive" : 0,
            "Non Invasive" : 1
        }

        # taxa_folders = [d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d)) and d != "metadata"]
        # for taxon in taxa_folders:
        #     meta_path = os.path.join(root_dir, "filtered_metadata", f"{taxon}_metadata.csv")
        #     if not os.path.exists(meta_path):
        #         continue
        #     metadata_df = pd.read_csv(meta_path)
        #     for idx, row in metadata_df.iterrows():
        #       if pd.isna(row["filename"]):
        #           print(f"⚠️ Missing filename in taxon '{taxon}' at index {idx}")
        #           continue
        #       self.entries.append((taxon, row))

        for image in os.listdir(root_dir): #root_dir = '/content/drive/MyDrive/Thesis/images_with_holes/linear_opposite' or whatever subfolder we are in
          taxon = image.split('_')[0] + '_' + image.split('_')[1]
          row = {
              'filename' : image,
              'label' : "Invasive" if taxon in ['lythrum_salicaria', 'lythrum_virgatum'] else "Non Invasive"
          }

          self.entries.append((taxon, row))


    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        taxon, row = self.entries[idx]
        filename = str(row["filename"])
        image_path = os.path.join(self.root_dir, filename)
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = self.label_map.get(row["label"])
        return image, label, idx

In [ ]:
#@title Return the dataloader of all images and not already separated (also class weights are not present)
def get_data_loader(root_dir, batch_size=32, use_bioclip=True, preprocess=None):
    if use_bioclip and preprocess:
        transform = preprocess
    else:
        transform = transforms.Compose([
          transforms.Resize((224, 224)),
          transforms.ToTensor()
        ])

    dataset = InvasiveSpeciesDataset(
        root_dir=root_dir,
        transform=transform,
    )

    loader = DataLoader(dataset, batch_size=32, shuffle=False)

    return loader

In [ ]:
#@title 2.Define the embedding extractor:
#ResNet without the last FC layer
def get_resnet_embeddings_extractor_model():
    #Use a pre-trained ResNet18 model
    model = models.resnet18(weights='IMAGENET1K_V1')

    #Freeze all layers
    for param in model.parameters():
        param.requires_grad = False

    embeddings_extractor= nn.Sequential(*list(model.children())[:-1]) #removes the last FC layer
    print("Model obtained...")

    return embeddings_extractor

#BioCLIP model
def get_bioclip_embeddings_extractor_model(device, bioclip_version=2):
    if bioclip_version==2:
        print("BioCLIP 2!")
        model, _, preprocess = open_clip.create_model_and_transforms('hf-hub:imageomics/bioclip-2')
    else:
        model, _, preprocess = open_clip.create_model_and_transforms('hf-hub:imageomics/bioclip')
    print("Model obtained...")
    model.to(device)
    model.eval()
    return model, preprocess

In [ ]:
#@title 3. Extract embeddings using the embeddings_extractor and return the embeddings tensors
def extract_embeddings(dataloader, embeddings_extractor, device, embeddings_save_path, augmented=False, use_bioclip=True, dataset=None):
    embeddings_extractor.eval()
    all_embeddings = []
    all_labels = []
    all_ids = []
    all_groups = []  # ✅ NEW: store taxa names
    all_filenames = [] #store filenames

    with torch.no_grad():
      for images, labels, obs_id, taxon in tqdm(dataloader, desc='Extracting embeddings'):
          images = images.to(device)
          if use_bioclip:
              embeddings = embeddings_extractor.encode_image(images)
              embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)  # Optional: normalize
          else:
              embeddings = embeddings_extractor(images).view(images.size(0), -1)

          all_embeddings.append(embeddings.cpu())
          all_labels.append(labels)
          all_ids.extend(obs_id)

          # get filenames aligned with obs_id
          batch_filenames = [dataloader.dataset.entries[int(i)][1]["filename"] for i in obs_id]
          all_filenames.extend(batch_filenames)

          all_groups.extend(taxon) #remember to use extend and not append to get a flat list and not a list of tuples of the batch

          # # ✅ OLD: reconstruct taxa names directly here
          # if dataset is not None:
          #     for i in obs_id:
          #         taxon, _ = dataset.entries[int(i)]
          #         all_groups.append(taxon)

    embeddings_tensors = torch.cat(all_embeddings, dim=0).to(device)
    labels_tensors = torch.cat(all_labels, dim=0).to(device)
    obs_id_tensor = torch.tensor(all_ids).to(device)

    all_embeddings = {
       'embeddings' : embeddings_tensors,
       'labels' : labels_tensors,
       'ids' : obs_id_tensor,
       'groups': all_groups,  # ✅ Save group names
       'filenames' : all_filenames
    }

    if augmented==False:
      torch.save(all_embeddings, embeddings_save_path)

    # embedding_size = embeddings_tensors.shape[1] #useful for classifier

    return all_embeddings

In [ ]:
#@title Split the embeddings 80-20, then add the augmentation to the training set

### OLD FUNCTION WHERE REMOVE_SPECIES WERE REMOVED FROM BOTH TRAINING SET AND VALIDATION SET
def split_embeddings_1(embeddings_dict, augmented_embeddings_dict=None, val_percentage=0.2, remove_species=None):

    """Split the embeddings and not the images into training set and validation set"""

    embeddings_tensors = embeddings_dict["embeddings"]
    labels_tensors = embeddings_dict["labels"]
    obs_id_tensor = embeddings_dict["ids"]


    #modified to remove species
    embeddings_np = embeddings_tensors.cpu().numpy()
    labels_np = labels_tensors.cpu().numpy()
    obs_id_np = obs_id_tensor.cpu().numpy()
    groups_np = np.array(embeddings_dict["groups"])  # ✅ Directly use stored groups

    if remove_species != None:
      #mask
      indices = [groups_np[i] not in remove_species for i in range(len(groups_np))]

      embeddings_np_reduced = np.array([embeddings_np[i] for i in range(len(indices)) if indices[i]])
      labels_np_reduced = np.array([labels_np[i] for i in range(len(indices)) if indices[i]])
      obs_id_np_reduced = np.array([obs_id_np[i] for i in range(len(indices)) if indices[i]])
      groups_np_reduced = [groups_np[i] for i in range(len(indices)) if indices[i]]

    else:
      embeddings_np_reduced = embeddings_np
      labels_np_reduced = labels_np
      obs_id_np_reduced = obs_id_np
      groups_np_reduced = groups_np

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # or get from caller
    embeddings_tensors = torch.from_numpy(np.stack(embeddings_np_reduced)).to(device)
    labels_tensors = torch.from_numpy(np.array(labels_np_reduced)).to(device)
    obs_id_tensor = torch.from_numpy(np.array(obs_id_np_reduced)).to(device)
    #end of modification

    embedding_size = embeddings_tensors.shape[1] #useful for classifier
    dataset = torch.utils.data.TensorDataset(embeddings_tensors, labels_tensors, obs_id_tensor)

    train_size = int((1-val_percentage) * len(dataset))
    val_size = len(dataset) - train_size

    # ✅ make split deterministic
    g = torch.Generator().manual_seed(42)
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=g)

    #if using augmentation
    if augmented_embeddings_dict:
      print("Augmenting train dataset...")
      aug_embeddings_tensors = augmented_embeddings_dict["embeddings"]
      aug_labels_tensors = augmented_embeddings_dict["labels"]
      aug_obs_id_tensor = augmented_embeddings_dict["ids"]

      #modification to remove species
      aug_embeddings_np = aug_embeddings_tensors.cpu().numpy()
      aug_labels_np = aug_labels_tensors.cpu().numpy()
      aug_obs_id_np = aug_obs_id_tensor.cpu().numpy()
      aug_groups_np = np.array(augmented_embeddings_dict["groups"])  # ✅ Directly use stored groups

      if remove_species != None:
        indices = [aug_groups_np[i] not in remove_species for i in range(len(aug_groups_np))]

        #this is to remove species (e.g hyssopifolia) from augmentation
        aug_embeddings_np_reduced = np.array([aug_embeddings_np[i] for i in range(len(indices)) if indices[i]])
        aug_labels_np_reduced = np.array([aug_labels_np[i] for i in range(len(indices)) if indices[i]])
        aug_obs_id_np_reduced = np.array([aug_obs_id_np[i] for i in range(len(indices)) if indices[i]])
        aug_groups_np_reduced = np.array([aug_groups_np[i] for i in range(len(indices)) if indices[i]])


      else:
        aug_embeddings_np_reduced = aug_embeddings_np
        aug_labels_np_reduced = aug_labels_np
        aug_obs_id_np_reduced = aug_obs_id_np
        aug_groups_np_reduced = aug_groups_np

      device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # or get from caller
      aug_embeddings_tensors = torch.from_numpy(np.stack(aug_embeddings_np_reduced)).to(device)
      aug_labels_tensors = torch.from_numpy(np.array(aug_labels_np_reduced)).to(device)
      aug_obs_id_tensor = torch.from_numpy(np.array(aug_obs_id_np_reduced)).to(device)

      #end of modification
      augmented_train_dataset = torch.utils.data.TensorDataset(aug_embeddings_tensors, aug_labels_tensors, aug_obs_id_tensor)

      #concatenate train dataset with augmented train dataset
      train_dataset = ConcatDataset([train_dataset, augmented_train_dataset])

    label_counts = Counter()
    for _, label, _ in train_dataset:
        label_counts[int(label)] += 1

    print("🔢 Class distribution in train set:", label_counts)

    # total = sum(label_counts.values())
    class_weights = torch.tensor([
        1.0 / math.log(1.02 + label_counts[0]),
        1.0 / math.log(1.02 + label_counts[1])
    ], dtype=torch.float)

    print("⚖️ Class weights:", class_weights)

    g = torch.Generator()
    g.manual_seed(42)

    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True, generator=g)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)

    return train_loader, val_loader, class_weights, embedding_size

### NEW VERSION WHERE I RETAIN THE SPECIES IN REMOVE_SPECIES IN THE VALIDATION SPLIT
def split_embeddings_2(embeddings_dict, augmented_embeddings_dict=None, val_percentage=0.2, remove_species=None):
    """Split the embeddings into training and validation sets, then remove specific species only from training."""

    embeddings_tensors = embeddings_dict["embeddings"]
    labels_tensors = embeddings_dict["labels"]
    obs_id_tensor = embeddings_dict["ids"]
    groups_np = np.array(embeddings_dict["groups"])  # Taxa names

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    dataset = torch.utils.data.TensorDataset(embeddings_tensors, labels_tensors, obs_id_tensor)
    full_indices = np.arange(len(dataset))

    # Shuffle and split
    g = torch.Generator()
    g.manual_seed(42)

    perm = torch.randperm(len(full_indices), generator=g)
    split = int(len(full_indices) * (1 - val_percentage))
    train_idx = perm[:split]
    val_idx   = perm[split:]

    train_dataset = torch.utils.data.Subset(dataset, train_idx)
    val_dataset   = torch.utils.data.Subset(dataset, val_idx)

    # Filter remove_species only from training set
    if remove_species is not None:
        train_indices = train_dataset.indices if hasattr(train_dataset, 'indices') else train_dataset
        filtered_train_data = [
            dataset[i] for i in train_indices
            if groups_np[i] not in remove_species
        ]
        train_dataset = torch.utils.data.TensorDataset(
            torch.stack([item[0] for item in filtered_train_data]),
            torch.stack([item[1] for item in filtered_train_data]),
            torch.stack([item[2] for item in filtered_train_data])
        )

    # Augmented embeddings (optional)
    if augmented_embeddings_dict:
        print("Augmenting train dataset...")
        aug_embeddings = augmented_embeddings_dict["embeddings"]
        aug_labels = augmented_embeddings_dict["labels"]
        aug_ids = augmented_embeddings_dict["ids"]
        aug_groups = np.array(augmented_embeddings_dict["groups"])

        if remove_species is not None:
            keep_mask = np.array([g not in remove_species for g in aug_groups])
            aug_embeddings = aug_embeddings[keep_mask]
            aug_labels = aug_labels[keep_mask]
            aug_ids = aug_ids[keep_mask]

        aug_dataset = torch.utils.data.TensorDataset(aug_embeddings, aug_labels, aug_ids)
        train_dataset = ConcatDataset([train_dataset, aug_dataset])

    # Compute class weights
    label_counts = Counter()
    for _, label, _ in train_dataset:
        label_counts[int(label)] += 1

    print("🔢 Class distribution in train set:", label_counts)

    class_weights = torch.tensor([
        1.0 / math.log(1.02 + label_counts[0]),
        1.0 / math.log(1.02 + label_counts[1])
    ], dtype=torch.float)

    print("⚖️ Class weights:", class_weights)

    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True, generator=g)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)

    embedding_size = embeddings_tensors.shape[1]
    return train_loader, val_loader, class_weights, embedding_size

In [ ]:
#@title Classifier
class InvasiveSpeciesClassifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dim=256, num_classes=2):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(), #consider adding a Dropout layer
            # nn.Dropout(0.3), #We remove the dropout layer
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.classifier(x)

In [ ]:
#@title 5. Evaluation function
def evaluate_classifier(loader, dataset, feature_extractor, classifier, criterion, device, save_csv_path, embeddings_dict):
    print("---inside evaluation function---")
    classifier.eval()
    total_loss, correct, total = 0, 0, 0

    all_preds = []
    all_labels = []
    all_obs_ids = []
    all_filenames = []
    all_groups = []

    # ✅ NEW: store raw logits and softmax probabilities
    all_logits = []
    all_probs = []
    softmax = torch.nn.Softmax(dim=1)


    with torch.no_grad():
        for embeddings, labels, obs_ids in tqdm(loader, desc="Evaluating"): # Unpack all three values
            embeddings, labels = embeddings.to(device), labels.to(device)

            outputs = classifier(embeddings)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * embeddings.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_obs_ids.extend(obs_ids.cpu().numpy() if torch.is_tensor(obs_ids) else obs_ids)

            # ✅ fetch filenames and groups directly from embeddings_dict
            if embeddings_dict is not None:
                fnames = [embeddings_dict["filenames"][int(oid)] for oid in obs_ids]
                taxa   = [embeddings_dict["groups"][int(oid)] for oid in obs_ids]
                all_filenames.extend(fnames)
                all_groups.extend(taxa)
            else:
                all_filenames.extend([None] * len(obs_ids))
                all_groups.extend([None] * len(obs_ids))

            # ✅ NEW: capture logits and probabilities for this batch
            logits_b = outputs.detach().cpu()
            probs_b  = softmax(outputs).detach().cpu()

            all_logits.extend(logits_b.tolist())
            all_probs.extend(probs_b.tolist())

    avg_loss = total_loss / total
    avg_acc = correct / total

    if save_csv_path:
        # ✅ wide-format CSV: separate columns for each class’s logits/probs
        n_classes = len(all_probs[0]) if len(all_probs) else 0
        base = {
            "obs_id": all_obs_ids,
            "filename": all_filenames,
            "group": all_groups,
            "y_true": all_labels,
            "y_pred": all_preds,
        }
        for i in range(n_classes):
            base[f"logit_c{i}"] = [row[i] for row in all_logits]
            base[f"prob_c{i}"]  = [row[i] for row in all_probs]

        df = pd.DataFrame(base)
        df.to_csv(save_csv_path, index=False, mode='a')  # append instead of overwriting
        print(f"📂 Saved evaluation results to {save_csv_path}")

    return avg_loss, avg_acc, all_labels, all_preds, all_obs_ids, all_filenames

In [ ]:
#@title Retrieve specific model

#•create a dummy image to take the embedding dimensions
def get_embedding_dim(embedding_model, preprocess, device):
    dummy_image = Image.new('RGB', (224, 224), color='white')
    image_tensor = preprocess(dummy_image).unsqueeze(0).to(device)
    with torch.no_grad():
        if hasattr(embedding_model, 'encode_image'):
            embedding = embedding_model.encode_image(image_tensor)
        else:
            embedding = embedding_model(image_tensor)

    embedding_dim = embedding.shape[1]
    return embedding_dim

def define_each_model(feature_extractor, preprocess, classifier_folder, device):
    models_dict = {}

    for classifier_path in os.listdir(classifier_folder):
        if classifier_path.endswith(".pth"):
            species_name = classifier_path.replace(".pth", "").split("_")[-2] + "_" + classifier_path.replace(".pth", "").split("_")[-1]

            classifier_path = os.path.join(classifier_folder, classifier_path)
            classifier = InvasiveSpeciesClassifier(embedding_dim=get_embedding_dim(feature_extractor, preprocess, device))
            classifier.load_state_dict(torch.load(classifier_path, map_location=device))
            classifier.eval().to(device)

            model = FullModel(feature_extractor, classifier).to(device)
            model.eval()

            models_dict[species_name] = model

    return models_dict

In [ ]:
#@title Full Model
class FullModel(nn.Module):
    def __init__(self, embedding_model, classifier_head):
        super().__init__()
        self.embedding_model = embedding_model
        self.classifier_head = classifier_head

    def forward(self, x):
        if hasattr(self.embedding_model, 'encode_image'):
            features = self.embedding_model.encode_image(x)
        else:
            features = self.embedding_model(x)

        if features.ndim == 4:
            features = features.view(features.size(0), -1)

        return self.classifier_head(features)

In [ ]:
#@title Load model and predict
def load_model_and_predict(folder_name="taxas", use_bioclip=True, bioclip_version=None,  loss_type='CE', output_dir="invasive_species",
                             model_name="prova", verbose=True, save_model=True, embeddings_save_path=None,
                             epochs=50, debug_subset_size=None, do_early_stopping=False, remove_species=None,
                             keep_hys_in_val = False, full_model_path = None, classifier_path = None, predictions_path = None):


    # random.seed(42);
    np.random.seed(42)
    torch.manual_seed(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    start_time = time.time()

    github = '/content/ai4biological-pattern/barbara_new'
    github_repository = f'{github}/{folder_name}'

    if not os.path.isdir(github_repository): #check if data has already been downloaded
        !git sparse-checkout set barbara_new/{folder_name}
        !git checkout clean-barbara

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    if use_bioclip:
        feature_extractor, preprocess = get_bioclip_embeddings_extractor_model(device, bioclip_version)
        feature_extractor = feature_extractor.to(device)
    else:
        feature_extractor = get_resnet_embeddings_extractor_model()
        feature_extractor = feature_extractor.to(device)
        preprocess = None
    print("Feature extractor obtained..")

    dataset = InvasiveSpeciesDataset(root_dir=github_repository, transform=preprocess if use_bioclip else None)

    if os.path.exists(embeddings_save_path):

        print("✅ Embeddings already exist, loading from file...")
        embeddings_dict = torch.load(embeddings_save_path)

        img_loader = get_data_loader(root_dir=github_repository)

        augmented_embeddings_dict=None

    else:

        img_loader = get_data_loader(root_dir=github_repository)
        print("Dataloader obtained..")

        #this function also saves embeddings in the desired path
        #input size is the same for training and validation, is is the size (number of features) of the single embedding.
        embeddings_dict = extract_embeddings(dataloader=img_loader, embeddings_extractor=feature_extractor, device=device, embeddings_save_path=embeddings_save_path, use_bioclip=use_bioclip, dataset=dataset)


        augmented_embeddings_dict=None


    classifier = InvasiveSpeciesClassifier(embedding_dim=input_size).to(device)

    #after initialization, load finetuned classifier
    print("Loading classifier...")
    classifier.load_state_dict(torch.load(classifier_path))
    print("Classifier loaded")

    if loss_type=='CE':
      criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))


    # full_model = FullModel(feature_extractor, classifier)
    # #after initialization, load finetuned model
    # print("Loading full model parameters...")
    # full_model.load_state_dict(torch.load(full_model_path))
    # print("Full model obtained...")
    # full_model.eval()

    avg_loss, avg_acc, all_labels, all_preds, all_obs_ids, all_filenames = evaluate_classifier(embeddings_val_loader, dataset, feature_extractor, classifier, criterion, device, predictions_path, embeddings_dict)

    #just to check
    print("number of different elements: ", sum(1 for i, j in zip(all_labels, all_preds) if i != j))
    print("accuracy: ", 1-(sum(1 for i, j in zip(all_labels, all_preds) if i != j))/len(all_preds))
    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Total execution time: {elapsed_time:.2f} seconds")

    #return the validation loss and accuracy for the last epoch
    return

In [ ]:
#@title Predict for each trait pair folders

def predict_trait_pairs(folder_name="taxas",use_bioclip=True,bioclip_version=2, subfolder=None, predictions_path=None):
    torch.cuda.empty_cache()

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    github = '/content/ai4biological-pattern/barbara_new'
    github_repository = f'{github}/{folder_name}'

    # if not os.path.isdir(github_repository):
    !git sparse-checkout set barbara_new barbara_new/dataset_label_patter barbara_new/taxas
    !git checkout clean-barbara

    classifier_folder = f'{github_repository}/leave_one_out_models'

    feature_extractor, preprocess = get_bioclip_embeddings_extractor_model(device, bioclip_version=bioclip_version)
    feature_extractor.eval().to(device)

    models_dict = define_each_model(feature_extractor, preprocess, classifier_folder, device)
    print(f"{len(models_dict)} models loaded...")

    pair_subfolder = f'/content/drive/MyDrive/Thesis/images_with_holes/{subfolder}'

    dataset = InvasiveSpeciesDataset(root_dir=pair_subfolder, transform=preprocess)
    print(f"Dataset loaded with {len(dataset)} images")

    results = []  # ✅ collect all results here

    indices = [i for i in range(len(dataset))]

    for i in tqdm(indices, desc="Processing images"):

        image_tensor, label, idx = dataset[i]

        taxon, row = dataset.entries[idx]

        filename = row["filename"]

        model = models_dict.get(taxon)
        if model is None:
            print(f"No model found for {taxon}")
            continue


        input_tensor = image_tensor.unsqueeze(0).to(device)

        with torch.no_grad():
          output = model(input_tensor)               # logits
          probs = F.softmax(output, dim=1)           # probabilities
          pred_class = torch.argmax(probs, dim=1).item()

          logits = output[0].detach().cpu().tolist()
          probs_ = probs[0].detach().cpu().tolist()

        results.append({
            "filename": filename,
            "group": taxon,
            "y_true": int(label) if torch.is_tensor(label) else label,
            "y_pred": pred_class,
            "logit_c0": logits[0],
            "prob_c0": probs_[0],
            "logit_c1": logits[1],
            "prob_c1": probs_[1],
        })


# ✅ Save results to CSV
    df = pd.DataFrame(results)
    if predictions_path:
        df.to_csv(predictions_path, index=False)
        print(f"📂 Saved predictions to {predictions_path}")

    return df



## Predicting

In [ ]:
pairs_folder = '/content/drive/MyDrive/Thesis/images_with_holes'


for pair in os.listdir(pairs_folder):

  if pair != 'linear_opposite': #because I had already done it previously
    predictions_path = f'/content/drive/MyDrive/Thesis/predictions_masked_regions_{pair}.csv'

    predict_trait_pairs(folder_name="taxas",use_bioclip=True,bioclip_version=2, subfolder=pair, predictions_path=predictions_path)

Updating files: 100% (45650/45650), done.
Already on 'clean-barbara'
Your branch is up to date with 'origin/clean-barbara'.
BioCLIP 2!


open_clip_config.json:   0%|          | 0.00/534 [00:00<?, ?B/s]

open_clip_model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Model obtained...
30 models loaded...
Dataset loaded with 24083 images


Processing images: 100%|██████████| 24083/24083 [1:23:42<00:00,  4.79it/s]


...

📂 Saved predictions to /content/drive/MyDrive/Thesis/predictions_masked_regions_erect_sessile.csv
Already on 'clean-barbara'
Your branch is up to date with 'origin/clean-barbara'.
BioCLIP 2!
Model obtained...
30 models loaded...
Dataset loaded with 21567 images


Processing images: 100%|██████████| 21567/21567 [1:26:05<00:00,  4.18it/s]


📂 Saved predictions to /content/drive/MyDrive/Thesis/predictions_masked_regions_erect_opposite.csv
Already on 'clean-barbara'
Your branch is up to date with 'origin/clean-barbara'.
BioCLIP 2!
Model obtained...
30 models loaded...
Dataset loaded with 2693 images


Processing images: 100%|██████████| 2693/2693 [08:59<00:00,  4.99it/s]

📂 Saved predictions to /content/drive/MyDrive/Thesis/predictions_masked_regions_alternate_subsessile.csv
